# Experiment 4: Text-to-Image Generation using a Pre-trained Diffusion Model

**Aim:** Generate images from natural-language prompts using the pre-trained Stable Diffusion XL model and a Gradio interface.

> Recommended: use a Colab GPU runtime. The model is large and CPU generation can be very slow.


In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors gradio

In [ ]:
import torch
from diffusers import DiffusionPipeline
import gradio as gr

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Load the pre-trained SDXL model

The first run downloads model files, so allow time for setup. We use half precision on a CUDA GPU to reduce memory use.


In [ ]:
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    use_safetensors=True,
    variant="fp16" if torch.cuda.is_available() else None,
)

pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")
print("Model loaded.")


## 2. Generate a test image

Change the prompt and rerun this cell to compare results.


In [ ]:
prompt = "A tiny glass greenhouse on Mars at sunrise, cinematic lighting, highly detailed"
negative_prompt = "blurry, low quality, distorted"

generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(42)
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=25,
    guidance_scale=7.0,
    generator=generator,
)
image = result.images[0]
display(image)


## 3. Create a Gradio interface

Enter a prompt, adjust inference steps and guidance scale, and click **Generate Image**. The notebook uses `share=True` to request a shareable Gradio link; only share it with people you trust.


In [ ]:
def generate_image(prompt, negative_prompt, steps, guidance_scale, seed):
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("Please enter a text prompt first.")

    generator = None
    if seed is not None and int(seed) >= 0:
        generator = torch.Generator(
            device="cuda" if torch.cuda.is_available() else "cpu"
        ).manual_seed(int(seed))

    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt or None,
        num_inference_steps=int(steps),
        guidance_scale=float(guidance_scale),
        generator=generator,
    )
    return result.images[0]

with gr.Blocks(title="SDXL Text-to-Image Generator") as demo:
    gr.Markdown("# Text-to-Image Generation using Stable Diffusion XL")
    prompt_box = gr.Textbox(
        label="Prompt",
        placeholder="Describe the image you want to create...",
        lines=3,
    )
    negative_box = gr.Textbox(
        label="Negative prompt (optional)",
        placeholder="blurry, low quality, distorted...",
        lines=2,
    )
    with gr.Row():
        steps_slider = gr.Slider(10, 50, value=25, step=1, label="Inference steps")
        guidance_slider = gr.Slider(1, 12, value=7, step=0.5, label="Guidance scale")
    seed_box = gr.Number(value=-1, precision=0, label="Seed (-1 for random)")
    output_image = gr.Image(label="Generated image", type="pil")
    generate_button = gr.Button("Generate Image", variant="primary")
    generate_button.click(
        generate_image,
        inputs=[prompt_box, negative_box, steps_slider, guidance_slider, seed_box],
        outputs=output_image,
    )

demo.launch(share=True)


## 4. Observations

Record what changes when you modify one parameter at a time:

| Test | What to vary | What to observe |
|---|---|---|
| Prompt | Subject, style, lighting, setting | Relevance and visual style |
| Inference steps | For example, 15 vs 30 | Detail vs generation time |
| Guidance scale | For example, 4 vs 8 | Prompt adherence and image quality |
| Negative prompt | Add/remove unwanted traits | Whether unwanted features are reduced |

## Conclusion
This experiment demonstrates text-to-image generation using a pre-trained diffusion model. It introduces prompt engineering, inference steps, guidance scale, Hugging Face Diffusers, and an interactive Gradio interface.

## Troubleshooting
- **CUDA out of memory:** Restart the runtime, ensure GPU is enabled, close other GPU sessions, and try fewer inference steps. If needed, use a smaller model or memory-saving offload settings.
- **Model download is slow:** The initial model download can be large; wait for it to complete.
- **Gradio launch issue:** Rerun the installation cell and restart the runtime if package versions conflict.
- **No GPU:** CPU inference may be extremely slow for SDXL.
